# Linear Regression

In [2]:
# Import the required libraries

import numpy as np
import matplotlib.pyplot as plt
import math, copy
import pandas as pd
import matplotlib.pyplot as plt


## Data Preprocessing

### **Exploring the dataset**

Let's start with loading the training data from the csv into a pandas dataframe



Load the datasets from GitHub. Train dataset has already been loaded for you in df below. To get test dataset use the commented code.

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/train_processed_splitted.csv')

Let's see what the first 5 rows of this dataset looks like

In [4]:
df.head()

,LotArea,TotalBsmtSF,GrLivArea,GarageArea,PoolArea,OverallCond,Utilities,SalePrice
0,11553,1051,1159,336,0,5,AllPub,158000
1,8400,1052,1052,288,0,5,AllPub,138500
2,8960,1008,1028,360,0,6,AllPub,115000
3,11100,0,930,308,0,7,AllPub,84900
4,15593,1304,2287,667,0,4,AllPub,225000


What are all the features present? What is the range for each of the features along with their mean?

In [ ]:
print(df.columns.tolist())
stats = df.describe().loc[['min', 'max', 'mean']].T
stats['range'] = stats['max'] - stats['min']
print(stats)
print(df['Utilities'].unique())

### **Feature Scaling and One-Hot Encoding**

You must have noticed that some features `(such as Utilities)` are not continuous values.
  
These features contain values indicating different categories and must somehow be converted to numbers so that the computer can understand it. `(Computers only understand numbers and not strings)`
  
These features are called categorical features. We can represent these features as a `One-Hot Representation`
  
  
You must have also noticed that all the other features, each are in a different scale. This can be detremental to the performance of our linear regression model and so we normalize them so that all of them are in the range $[0,1]$

> NOTE: When you are doing feature scaling, store the min/max which you will use to normalize somewhere. This is then to be used at testing time. Try to think why are doing this?

In [ ]:
df = df.join(pd.get_dummies(df['Utilities'], dtype='int32', prefix='Utilities'))
df = df.drop('Utilities', axis=1)
df.head()

In [ ]:
min_max = {}
for col in df.columns:
    min_max[col] = (df[col].min(), df[col].max())
    df[col] = (df[col] - min_max[col][0]) / (min_max[col][1] - min_max[col][0])
df.head()

### **Conversion to NumPy**

Ok so now that we have all preprocessed all the data, we need to convert it to numpy for our linear regression model
  
Assume that our dataset has a total of $N$ datapoints. Each datapoint having a total of $D$ features (after one-hot encoding), we want our numpy array to be of shape $(N, D)$

In our task, we have to predict the `SalePrice`. We will need 2 numpy arrays $

*   List item
*   List item

(X, Y)$. These represent the features and targets respectively

In [ ]:
x = df.drop('SalePrice', axis=1).to_numpy()
y = df['SalePrice'].to_numpy().reshape(-1, 1)
print(x.shape, y.shape)

## Linear Regression formulation
  
We now have our data in the form we need. Let's try to create a linear model to get our initial (Really bad) prediction


Let's say a single datapoint in our dataset consists of 3 features $(x_1, x_2, x_3)$, we can pose it as a linear equation as follows:
$$ y = w_1x_1 + w_2x_2 + w_3x_3 + b $$
Here we have to learn 4 parameters $(w_1, w_2, w_3, b)$
  
  
Now how do we extend this to multiple datapoints?  
  
  
Try to answer the following:
- How many parameters will we have to learn in the cae of our dataset? (Don't forget the bias term)
- Form a linear equation for our dataset. We need just a single matrix equation which correctly represents all the datapoints in our dataset
- Implement the linear equation as an equation using NumPy arrays (Start by randomly initializing the weights from a standard normal distribution)

In [ ]:
w = np.random.randn(1, x.shape[1])
b = np.random.randn()
y_pred = x @ w.T + b

How well does our model perform? Try comparing our predictions with the actual values

In [ ]:
print(np.hstack((y_pred[:5], y[:5])))

### **Learning weights using gradient descent**

So these results are really horrible. We need to somehow update our weights so that it correclty represents our data. How do we do that?

We must do the following:
- We need some numerical indication for our performance, for this we define a Loss Function ( $\mathscr{L}$ )
- Find the gradients of the `Loss` with respect to the `Weights`
- Update the weights in accordance to the gradients: $W = W - \alpha\nabla_W \mathscr{L}$

Lets define the loss function:
- We will use the MSE loss since it is a regression task. (Specify the assumptions we make while doing so as taught in the class).
- Implement this loss as a function. (Use numpy as much as possible)

In [ ]:
def mse_loss_fn(y_true, y_pred):
    m = y_true.shape[0]
    return (1 / (2 * m)) * np.sum((y_pred - y_true) ** 2)

Calculate the gradients of the loss with respect to the weights (and biases). First write the equations down on a piece of paper, then proceed to implement it

In [ ]:
def get_gradients(y_true, y_pred, W, b, X):
    m = X.shape[0]
    e = y_pred - y_true
    dW = (1 / m) * (e.T @ X)
    db = (1 / m) * np.sum(e)
    return dW, db

Update the weights using the gradients

In [ ]:
def update(weights, bias, gradients_weights, gradients_bias, lr):
    weights = weights - lr * gradients_weights
    bias = bias - lr * gradients_bias
    return weights, bias

Put all these together to find the loss value, its gradient and finally updating the weights in a loop. Feel free to play around with different learning rates and epochs
  
> NOTE: The code in comments are just meant to be used as a guide. You will have to do changes based on your code

In [ ]:
NUM_EPOCHS = 1000
LEARNING_RATE = 2e-2

losses = []

for epoch in range(NUM_EPOCHS):
    y_pred = x @ w.T + b
    loss = mse_loss_fn(y, y_pred)
    losses.append(loss)
    dw, db = get_gradients(y, y_pred, w, b, x)
    w, b = update(w, b, dw, db, LEARNING_RATE)


Now use matplotlib to plot the loss graph

In [ ]:
plt.plot(losses)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

### **Testing with test data**

Load and apply all the preprocessing steps used in the training data for the testing data as well. Remember to use the **SAME** min/max values which you used for the training set and not recalculate them from the test set. Also mention why we are doing this.

To load test data from GitHub, use the code below.


In [ ]:
df_test = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/test_processed_splitted.csv')
print(df_test)

df_test = df_test.join(pd.get_dummies(df_test['Utilities'], dtype='int32', prefix='Utilities'))
df_test = df_test.drop('Utilities', axis=1)

missing_cols = set(df.columns) - set(df_test.columns)
for col in missing_cols:
    df_test[col] = 0

df_test = df_test[df.columns]

Using the weights learnt above, predict the values in the test dataset. Also answer the following questions:
- Are the predictions good?
- What is the MSE loss for the testset
- Is the MSE loss for testing greater or lower than training
- Why is this the case

In [ ]:
df_test.fillna(0, inplace=True)

for col in df_test.columns:
    df_test[col] = (df_test[col] - min_max[col][0]) / (min_max[col][1] - min_max[col][0])

print(df_test.isna().sum().sum())

x_test = df_test.copy().drop('SalePrice', axis=1).to_numpy()
y_test = df_test.copy()['SalePrice'].to_numpy().reshape(-1, 1)
print(x_test.shape)

In [ ]:
extra_cols = list(set(df_test.columns) - set(df.columns))
print("Extra columns in df_test:", extra_cols)

missing_cols = list(set(df.columns) - set(df_test.columns))
print("Missing columns in df_test:", missing_cols)

In [ ]:
y_pred_test = x_test @ w.T + b
loss_test = mse_loss_fn(y_pred_test, y_test)

p_min, p_max = min_max['SalePrice']
y_pred_test_scaled = y_pred_test * (p_max - p_min) + p_min
y_test_scaled = y_test * (p_max - p_min) + p_min

In [ ]:
idx = np.random.randint(0, x_test.shape[0], 5)
y_pred_test_sample = y_pred_test_scaled[idx].round().astype(int)
y_true_test_sample = y_test_scaled[idx].round().astype(int)

print('Predicted SalePrice: \t', y_pred_test_sample.squeeze().tolist())
print('Actual SalePrice: \t', y_true_test_sample.squeeze().tolist())
print('\nTest Loss: \t\t', loss_test)